<a href="https://colab.research.google.com/github/SAINIDHI2005/IDS_GNN_Repo/blob/main/graphSAGE/KNN_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATASET_PATH = "/content/drive/MyDrive/CIC_IoT_2023/Host_flow_temporal"

In [3]:
import os

print(os.listdir(DATASET_PATH))

['BENIGN.csv', 'DDoS.csv', 'Spoofing.csv', 'Recon.csv', 'Mirai.csv', 'DoS.csv']


In [4]:
!pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.8 MB/s eta 0:00:00


In [5]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

import torch
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv

from sklearn.neighbors import kneighbors_graph

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
ATTACK_FOLDERS = [

    "DDOS",
    "DOS",
    "MIRAI",
    "RECON",
    "SPOOFING"
]

In [7]:
benign_df = pd.read_csv(
    os.path.join(
        DATASET_PATH,
        "BENIGN.csv"
    )
)

benign_df = benign_df.sample(
    n=200000,
    random_state=42
)

benign_df["label"] = 0

print(len(benign_df))

200000


In [8]:
ATTACK_FILES = [
    "DDoS",
    "DoS",
    "Mirai",
    "Recon",
    "Spoofing"
]

attack_dfs = []

for attack in ATTACK_FILES:

    attack_df = pd.read_csv(
        os.path.join(
            DATASET_PATH,
            f"{attack}.csv"
        )
    )

    attack_df = attack_df.sample(
        n=40000,
        random_state=42
    )

    attack_df["label"] = 1

    attack_dfs.append(attack_df)

    print(attack, len(attack_df))

DDoS 40000
DoS 40000
Mirai 40000
Recon 40000
Spoofing 40000


In [9]:
malicious_df = pd.concat(
    attack_dfs,
    ignore_index=True
)

print(len(malicious_df))

200000


In [10]:
df = pd.concat(

    [benign_df, malicious_df],

    ignore_index=True
)

df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(df.shape)

(400000, 87)


In [11]:
print(df["label"].value_counts())

label
0    200000
1    200000
Name: count, dtype: int64


In [12]:
df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

df.fillna(0, inplace=True)

print(df.shape)

(400000, 87)


In [13]:
FEATURES_30 = [
    # paste your exact 30 features here
     "bidirectional_mean_ps",
    "dst2src_syn_packets",
    "src2dst_min_ps",
    "ip_version",
    "src2dst_max_piat_ms",
    "dst2src_stddev_piat_ms",
    "bidirectional_max_piat_ms",
    "bidirectional_rst_packets",
    "src2dst_max_ps",
    "src2dst_duration_ms",
    "src2dst_bytes",
    "src2dst_ece_packets",
    "bidirectional_max_ps",
    "dst2src_bytes",
    "bidirectional_bytes",
    "dst2src_max_ps",
    "bidirectional_min_ps",
    "bidirectional_syn_packets",
    "dst2src_stddev_ps",
    "bidirectional_ack_packets",
    "protocol",
    "dst_port",
    "bidirectional_mean_piat_ms",
    "dst2src_duration_ms",
    "src2dst_mean_ps",
    "src2dst_min_piat_ms",
    "bidirectional_stddev_ps",
    "src2dst_stddev_piat_ms",
    "bidirectional_cwr_packets",
    "src2dst_psh_packets"
]

X = df[FEATURES_30]

y = df["label"]

In [14]:
print("Number of features:", len(FEATURES_30))

Number of features: 30


In [15]:
missing = [f for f in FEATURES_30 if f not in df.columns]

print("Missing features:", missing)

Missing features: []


In [16]:
print(X.shape)
print(y.shape)

(400000, 30)
(400000,)


In [17]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
)

X_val = scaler.transform(
    X_val
)


In [19]:
GRAPH_SIZE = 1000

K = 15

In [20]:
def create_graph_dataset(X, y):

    graphs = []

    num_graphs = len(X) // GRAPH_SIZE

    for i in range(num_graphs):

        start = i * GRAPH_SIZE

        end = start + GRAPH_SIZE

        X_chunk = X[start:end]

        y_chunk = y.iloc[start:end]

        adjacency = kneighbors_graph(

            X_chunk,

            n_neighbors=K,

            mode='connectivity',

            include_self=False
        )

        edge_index = np.array(
            adjacency.nonzero()
        )

        edge_index = torch.tensor(
            edge_index,
            dtype=torch.long
        )

        x_tensor = torch.tensor(
            X_chunk,
            dtype=torch.float
        )

        y_tensor = torch.tensor(
            y_chunk.values,
            dtype=torch.long
        )

        graph = Data(

            x=x_tensor,

            edge_index=edge_index,

            y=y_tensor
        )

        graphs.append(graph)

    return graphs

In [21]:
train_graphs = create_graph_dataset(
    X_train,
    y_train
)

val_graphs = create_graph_dataset(
    X_val,
    y_val
)


In [22]:
train_loader = DataLoader(
    train_graphs,
    shuffle=True
)

val_loader = DataLoader(
    val_graphs,
    shuffle=False
)

In [23]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv


class GraphSAGE(torch.nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels,
        num_classes
    ):

        super().__init__()

        self.conv1 = SAGEConv(
            in_channels,
            hidden_channels
        )

        self.conv2 = SAGEConv(
            hidden_channels,
            hidden_channels
        )

        self.linear = torch.nn.Linear(
            hidden_channels,
            num_classes
        )

    def forward(
        self,
        x,
        edge_index
    ):

        # GraphSAGE Layer 1
        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.3,
            training=self.training
        )

        # GraphSAGE Layer 2
        x = self.conv2(
            x,
            edge_index
        )

        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.3,
            training=self.training
        )

        # Classification Layer
        x = self.linear(x)

        return x

In [24]:
device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'cpu'
)

print(f"Device Selected: {device}")

model = GraphSAGE(

    in_channels=X_train.shape[1],

    hidden_channels=256,

    num_classes=2

).to(device)

optimizer = torch.optim.Adam(

    model.parameters(),

    lr=0.001
)

criterion = torch.nn.CrossEntropyLoss()

Device Selected: cuda


In [25]:
def validate():

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for graph in val_loader:

            graph = graph.to(device)

            out = model(
                graph.x,
                graph.edge_index
            )

            pred = out.argmax(dim=1)

            correct += (pred == graph.y).sum().item()
            total += graph.y.size(0)

    return correct / total

In [26]:
import os
import time
import torch
import joblib

SAVE_DIR = "/content/drive/MyDrive/CIC_IoT_2023/Saved_Model"
os.makedirs(SAVE_DIR, exist_ok=True)

best_val_acc = 0.0
best_epoch = 0

start_time = time.time()

EPOCHS = 150

for epoch in range(EPOCHS):


    model.train()

    total_loss = 0

    for graph in train_loader:

        graph = graph.to(device)

        optimizer.zero_grad()

        out = model(
            graph.x,
            graph.edge_index
        )

        loss = criterion(
            out,
            graph.y
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    model.eval()

    train_correct = 0
    train_total = 0

    with torch.no_grad():

        for graph in train_loader:

            graph = graph.to(device)

            out = model(
                graph.x,
                graph.edge_index
            )

            pred = out.argmax(dim=1)

            train_correct += (pred == graph.y).sum().item()
            train_total += graph.y.size(0)

    train_acc = train_correct / train_total

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for graph in val_loader:

            graph = graph.to(device)

            out = model(
                graph.x,
                graph.edge_index
            )

            pred = out.argmax(dim=1)

            val_correct += (pred == graph.y).sum().item()
            val_total += graph.y.size(0)

    val_acc = val_correct / val_total

    if val_acc > best_val_acc:

        best_val_acc = val_acc
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            os.path.join(
                SAVE_DIR,
                "KNN_graphSAGE.pth"
            )
        )

        joblib.dump(
            scaler,
            os.path.join(
                SAVE_DIR,
                "KNN_scaler.pkl"
            )
        )

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1:03d}"
        f" | Loss {avg_loss:.4f}"
        f" | Train {train_acc:.4f}"
        f" | Validation {val_acc:.4f}"
    )

end_time = time.time()

training_time = end_time - start_time

print("\n" + "="*60)

print("Training Finished Successfully!")

print(f"Best Validation Accuracy : {best_val_acc:.4f}")
print(f"Best Epoch               : {best_epoch}")
print(f"Training Time            : {training_time:.2f} seconds")
print(f"Training Time            : {training_time/60:.2f} minutes")

print("\nBest model saved to:")
print(os.path.join(SAVE_DIR, "KNN_graphSAGE.pth"))

print("\nScaler saved to:")
print(os.path.join(SAVE_DIR, "KNN_scaler.pkl"))

print("="*60)

Epoch 001 | Loss 0.4382 | Train 0.8163 | Validation 0.8167
Epoch 002 | Loss 0.3808 | Train 0.8330 | Validation 0.8331
Epoch 003 | Loss 0.3606 | Train 0.8373 | Validation 0.8371
Epoch 004 | Loss 0.3484 | Train 0.8417 | Validation 0.8417
Epoch 005 | Loss 0.3391 | Train 0.8504 | Validation 0.8502
Epoch 006 | Loss 0.3310 | Train 0.8578 | Validation 0.8570
Epoch 007 | Loss 0.3250 | Train 0.8645 | Validation 0.8632
Epoch 008 | Loss 0.3201 | Train 0.8661 | Validation 0.8649
Epoch 009 | Loss 0.3145 | Train 0.8692 | Validation 0.8688
Epoch 010 | Loss 0.3109 | Train 0.8718 | Validation 0.8716
Epoch 011 | Loss 0.3062 | Train 0.8745 | Validation 0.8736
Epoch 012 | Loss 0.3029 | Train 0.8796 | Validation 0.8774
Epoch 013 | Loss 0.2992 | Train 0.8801 | Validation 0.8793
Epoch 014 | Loss 0.2967 | Train 0.8797 | Validation 0.8781
Epoch 015 | Loss 0.2939 | Train 0.8684 | Validation 0.8667
Epoch 016 | Loss 0.2911 | Train 0.8845 | Validation 0.8826
Epoch 017 | Loss 0.2899 | Train 0.8773 | Validation 0.87

In [27]:
import time
import torch

print("=" * 60)

# Training Time
print(f"Training Time: {training_time:.2f} seconds ({150} epochs)\n")

# GPU Memory
if torch.cuda.is_available():

    allocated = torch.cuda.memory_allocated() / (1024 ** 2)
    reserved = torch.cuda.memory_reserved() / (1024 ** 2)
    max_reserved = torch.cuda.max_memory_reserved() / (1024 ** 2)

    print(f"Allocated GPU Memory: {allocated:.2f} MB")
    print(f"Reserved GPU Memory: {reserved:.2f} MB")
    print(f"Max GPU Memory: {max_reserved:.2f} MB\n")

# Model Parameters
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters: {total_params}")
print(f"Trainable Parameters: {trainable_params}\n")

# Graph Memory
node_memory = (
    graph.x.element_size()
    * graph.x.nelement()
) / (1024 ** 2)

edge_memory = (
    graph.edge_index.element_size()
    * graph.edge_index.nelement()
) / (1024 ** 2)

graph_memory = node_memory + edge_memory

print(f"Node Feature Memory: {node_memory:.2f} MB")
print(f"Edge Memory: {edge_memory:.2f} MB")
print(f"Total Graph Memory: {graph_memory:.2f} MB")

print("=" * 60)

Training Time: 331.67 seconds (150 epochs)

Allocated GPU Memory: 19.88 MB
Reserved GPU Memory: 48.00 MB
Max GPU Memory: 48.00 MB

Total Parameters: 147458
Trainable Parameters: 147458

Node Feature Memory: 0.11 MB
Edge Memory: 0.23 MB
Total Graph Memory: 0.34 MB
